# Lab 2 - Feature Engineering + Improved Baseline

## Goal
Build an improved pre-race model for `is_top10` and compare it against Lab 1 baselines using the same temporal validation and the same primary metric.

## Constraints from the assignment
- At least 3 engineered features from F1 domain knowledge.
- At least 2 feature types among: lag, rolling aggregate, interaction, categorical encoding.
- One simple model only: Logistic Regression or Decision Tree.
- Temporal validation must match Lab 1 split boundaries.
- `RANDOM_SEED = 414` in all random states.

In [1]:
RANDOM_SEED = 414

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import fastf1

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

warnings.filterwarnings("ignore")
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 200)

print(f"RANDOM_SEED = {RANDOM_SEED}")

RANDOM_SEED = 414


In [2]:
YEARS = [2022, 2023, 2024]

rows = []
for year in YEARS:
    schedule = fastf1.get_event_schedule(year, include_testing=False)

    for round_number in schedule["RoundNumber"].dropna().astype(int).unique():
        try:
            session = fastf1.get_session(year, round_number, "R")
            session.load(laps=False, telemetry=False, weather=False, messages=False)

            race = session.results[[
                "Abbreviation",
                "FullName",
                "TeamName",
                "GridPosition",
                "Position",
            ]].copy()

            race["season"] = year
            race["round"] = round_number
            rows.append(race)
        except Exception as e:
            print(f"Could not load {year} round {round_number}: {e}")

df = pd.concat(rows, ignore_index=True)

df = df.rename(
    columns={
        "Abbreviation": "driver_code",
        "FullName": "driver_name",
        "TeamName": "team_name",
        "GridPosition": "grid_position",
        "Position": "finish_position",
    }
)

df["grid_position"] = pd.to_numeric(df["grid_position"], errors="coerce")
df["finish_position"] = pd.to_numeric(df["finish_position"], errors="coerce")

df = df[df["finish_position"].notna()].copy()
df["is_top10"] = (df["finish_position"] <= 10).astype(int)

df = df.sort_values(["season", "round", "driver_code"]).reset_index(drop=True)
print(df.shape)
df.head()

req         WARNING 	DEFAULT CACHE ENABLED! (2.78 MB) C:\Users\joaqu\AppData\Local\Temp\fastf1
core           INFO 	Loading data for Bahrain Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['16', '55', '44', '63', '20', '77', '31', '22', '14', '24', '47', '18', '23', '3', '4', '6', '27', '11', '1', '10']
core           INFO 	Loading data for Saudi Arabian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['1', '16', '55', '11', '63', '31', '4', '10', '20', '44', '24', '27', '18', '23', '77', '14', '3', '6', '22', '47']
core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driv

(1357, 8)


,driver_code,driver_name,team_name,grid_position,finish_position,season,round,is_top10
0,ALB,Alexander Albon,Williams,14.0,13.0,2022,1,0
1,ALO,Fernando Alonso,Alpine,8.0,9.0,2022,1,1
2,BOT,Valtteri Bottas,Alfa Romeo,6.0,6.0,2022,1,1
3,GAS,Pierre Gasly,AlphaTauri,10.0,20.0,2022,1,0
4,HAM,Lewis Hamilton,Mercedes,5.0,3.0,2022,1,1


## Temporal Validation (same as Lab 1)

- Train: 2022
- Validation: 2023
- Test: 2024

Primary metric: keep the same one justified in Lab 1 (fill this explicitly when you finalize the report).

In [3]:
train_df = df[df["season"] == 2022].copy()
val_df = df[df["season"] == 2023].copy()
test_df = df[df["season"] == 2024].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

for split_name, split_df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(split_name, split_df["is_top10"].value_counts(normalize=True).round(3).to_dict())

Train: (439, 8)
Validation: (439, 8)
Test: (479, 8)
train {1: 0.501, 0: 0.499}
val {1: 0.501, 0: 0.499}
test {1: 0.501, 0: 0.499}


## Feature 1 (Lag): `prev_finish_position`

Justification: a driver's immediately previous finish often captures short-term form and reliability.

Leakage guard: uses only past races through `groupby(...).shift(1)`, so the current race outcome is never used in its own prediction.

In [4]:
df = df.sort_values(["driver_code", "season", "round"]).reset_index(drop=True)
df["prev_finish_position"] = df.groupby("driver_code")["finish_position"].shift(1)

df[["season", "round", "driver_code", "finish_position", "prev_finish_position"]].head(12)

,season,round,driver_code,finish_position,prev_finish_position
0,2022,1,ALB,13.0,NaN
1,2022,2,ALB,14.0,13.0
2,2022,3,ALB,10.0,14.0
3,2022,4,ALB,11.0,10.0
4,2022,5,ALB,9.0,11.0
5,2022,6,ALB,18.0,9.0
6,2022,7,ALB,18.0,18.0
7,2022,8,ALB,12.0,18.0
8,2022,9,ALB,13.0,12.0
9,2022,10,ALB,20.0,13.0


## Feature 2 (Rolling): `avg_finish_last_3`

Justification: short rolling history stabilizes noisy one-race outcomes while preserving recent performance signal.

Leakage guard: the rolling mean is shifted by one race (`shift(1)`), so it contains only information known before race start.

In [5]:
df["avg_finish_last_3"] = (
    df.groupby("driver_code")["finish_position"]
    .transform(lambda s: s.rolling(window=3, min_periods=1).mean().shift(1))
)

df[["season", "round", "driver_code", "prev_finish_position", "avg_finish_last_3"]].head(12)

,season,round,driver_code,prev_finish_position,avg_finish_last_3
0,2022,1,ALB,NaN,NaN
1,2022,2,ALB,13.0,13.000000
2,2022,3,ALB,14.0,13.500000
3,2022,4,ALB,10.0,12.333333
4,2022,5,ALB,11.0,11.666667
5,2022,6,ALB,9.0,10.000000
6,2022,7,ALB,18.0,12.666667
7,2022,8,ALB,18.0,15.000000
8,2022,9,ALB,12.0,16.000000
9,2022,10,ALB,13.0,14.333333


## Feature 3 (Categorical Encoding): `constructor_tier`

Justification: team strength is a structural driver of race outcomes; grouping teams into top/mid/back can add robust signal.

Leakage guard: each season uses constructor performance from the previous season only.

In [6]:
team_perf = (
    df.groupby(["season", "team_name"], as_index=False)["finish_position"]
    .mean()
    .rename(columns={"finish_position": "team_avg_finish"})
)

team_perf["rank_in_season"] = team_perf.groupby("season")["team_avg_finish"].rank(method="dense", ascending=True)

seasons = sorted(df["season"].unique())
constructor_tier = pd.Series("unknown", index=df.index, dtype="object")

for season in seasons:
    prev = team_perf[team_perf["season"] == season - 1].copy()

    if prev.empty:
        continue

    max_rank = int(prev["rank_in_season"].max())
    top_cut = max(3, int(np.ceil(max_rank * 0.3)))
    mid_cut = max(6, int(np.ceil(max_rank * 0.7)))

    prev["tier"] = np.select(
        [prev["rank_in_season"] <= top_cut, prev["rank_in_season"] <= mid_cut],
        ["top", "mid"],
        default="back",
    )

    tier_map = dict(zip(prev["team_name"], prev["tier"]))
    mask = df["season"] == season
    constructor_tier.loc[mask] = df.loc[mask, "team_name"].map(tier_map).fillna("unknown")

df["constructor_tier"] = constructor_tier

df[["season", "team_name", "constructor_tier"]].drop_duplicates().head(15)

,season,team_name,constructor_tier
0,2022,Williams,unknown
21,2023,Williams,back
43,2024,Williams,back
67,2022,Alpine,unknown
89,2023,Aston Martin,mid
111,2024,Aston Martin,mid
135,2024,Ferrari,top
136,2024,Haas F1 Team,back
138,2022,Alfa Romeo,unknown
160,2023,Alfa Romeo,mid


## Leakage Guard Checklist (for all new features)

Apply this checklist to document that each feature is pre-race only.

In [7]:
leakage_guard = pd.DataFrame(
    {
        "check": [
            "Uses only pre-race available columns",
            "No direct use of current-race finish_position as predictor",
            "Lag/rolling features include shift(1)",
            "No future seasons used to build current season feature",
            "Feature computed before split-specific model fit",
            "No target leakage through label-derived columns",
            "No post-race timing or telemetry included",
            "Null handling does not peek at validation/test targets",
            "Temporal split boundaries preserved",
            "Feature logic documented in notebook",
        ],
        "status": [
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
            "PASS",
        ],
        "notes": [
            "Features rely on grid/team/history only",
            "finish_position only used to build shifted historical signals",
            "Implemented for prev and rolling features",
            "constructor_tier uses previous season only",
            "Same feature recipe for all splits",
            "is_top10 not used in predictors",
            "No lap times, pit stops, race-day outcomes",
            "Imputation statistics taken from train split",
            "Train=2022, Val=2023, Test=2024",
            "Each feature includes justification + leakage note",
        ],
    }
)

leakage_guard

,check,status,notes
0,Uses only pre-race available columns,PASS,Features rely on grid/team/history only
1,No direct use of current-race finish_position ...,PASS,finish_position only used to build shifted his...
2,Lag/rolling features include shift(1),PASS,Implemented for prev and rolling features
3,No future seasons used to build current season...,PASS,constructor_tier uses previous season only
4,Feature computed before split-specific model fit,PASS,Same feature recipe for all splits
5,No target leakage through label-derived columns,PASS,is_top10 not used in predictors
6,No post-race timing or telemetry included,PASS,"No lap times, pit stops, race-day outcomes"
7,Null handling does not peek at validation/test...,PASS,Imputation statistics taken from train split
8,Temporal split boundaries preserved,PASS,"Train=2022, Val=2023, Test=2024"
9,Feature logic documented in notebook,PASS,Each feature includes justification + leakage ...


## Modeling (Logistic Regression)

Model choice is intentionally simple to isolate the value of feature engineering.

In [8]:
feature_cols = [
    "grid_position",
    "round",
    "prev_finish_position",
    "avg_finish_last_3",
    "constructor_tier",
]
cat_cols = ["constructor_tier"]
num_cols = [c for c in feature_cols if c not in cat_cols]

split_df = {
    "train": df[df["season"] == 2022].copy(),
    "val": df[df["season"] == 2023].copy(),
    "test": df[df["season"] == 2024].copy(),
}

train_medians = split_df["train"][num_cols].median()

for k in split_df:
    split_df[k][num_cols] = split_df[k][num_cols].fillna(train_medians)
    split_df[k][cat_cols] = split_df[k][cat_cols].fillna("unknown")

X_train = pd.get_dummies(split_df["train"][feature_cols], columns=cat_cols, drop_first=False)
X_val = pd.get_dummies(split_df["val"][feature_cols], columns=cat_cols, drop_first=False)
X_test = pd.get_dummies(split_df["test"][feature_cols], columns=cat_cols, drop_first=False)

X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

y_train = split_df["train"]["is_top10"].astype(int)
y_val = split_df["val"]["is_top10"].astype(int)
y_test = split_df["test"]["is_top10"].astype(int)

model = LogisticRegression(random_state=RANDOM_SEED, max_iter=2000, class_weight="balanced")
model.fit(X_train, y_train)


def evaluate(y_true, y_pred, y_score):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_score) if y_true.nunique() > 1 else np.nan,
    }

val_pred = model.predict(X_val)
val_score = model.predict_proba(X_val)[:, 1]

test_pred = model.predict(X_test)
test_score = model.predict_proba(X_test)[:, 1]

lab2_val_metrics = evaluate(y_val, val_pred, val_score)
lab2_test_metrics = evaluate(y_test, test_pred, test_score)

print("Lab 2 model - Validation (2023)")
print(pd.Series(lab2_val_metrics).round(4))
print()
print("Lab 2 model - Test (2024)")
print(pd.Series(lab2_test_metrics).round(4))

Lab 2 model - Validation (2023)
Accuracy     0.7768
Precision    0.7824
Recall       0.7682
F1           0.7752
ROC-AUC      0.8305
dtype: float64

Lab 2 model - Test (2024)
Accuracy     0.8267
Precision    0.8756
Recall       0.7625
F1           0.8151
ROC-AUC      0.9056
dtype: float64


## Baseline Comparison (Lab 1 vs Lab 2)

Baselines are evaluated on the same split and metrics to keep comparison fair.

In [9]:
def majority_baseline_metrics(train_target, eval_target):
    majority_class = int(train_target.mode().iloc[0])
    y_pred = np.full(shape=len(eval_target), fill_value=majority_class)
    y_score = np.full(shape=len(eval_target), fill_value=float(majority_class))
    return evaluate(eval_target, y_pred, y_score)


def domain_heuristic_metrics(eval_df):
    y_true = eval_df["is_top10"].astype(int)
    y_pred = (eval_df["grid_position"] <= 10).fillna(False).astype(int)

    # Higher score = more likely top-10; better grid is stronger score.
    score = -eval_df["grid_position"].fillna(eval_df["grid_position"].max() + 1)
    return evaluate(y_true, y_pred, score)


majority_val = majority_baseline_metrics(y_train, y_val)
heuristic_val = domain_heuristic_metrics(split_df["val"])

comparison_val = pd.DataFrame(
    {
        "Model / Baseline": [
            "Majority class (Lab 1)",
            "Domain heuristic (Lab 1)",
            "Lab 2 model (LogReg)",
        ],
        "Accuracy": [majority_val["Accuracy"], heuristic_val["Accuracy"], lab2_val_metrics["Accuracy"]],
        "Precision": [majority_val["Precision"], heuristic_val["Precision"], lab2_val_metrics["Precision"]],
        "Recall": [majority_val["Recall"], heuristic_val["Recall"], lab2_val_metrics["Recall"]],
        "F1": [majority_val["F1"], heuristic_val["F1"], lab2_val_metrics["F1"]],
        "ROC-AUC": [majority_val["ROC-AUC"], heuristic_val["ROC-AUC"], lab2_val_metrics["ROC-AUC"]],
    }
)

comparison_val_rounded = comparison_val.copy()
for col in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]:
    comparison_val_rounded[col] = comparison_val_rounded[col].round(4)

print("Validation comparison (2023)")
comparison_val_rounded

Validation comparison (2023)


,Model / Baseline,Accuracy,Precision,Recall,F1,ROC-AUC
0,Majority class (Lab 1),0.5011,0.5011,1.0000,0.6677,0.5000
1,Domain heuristic (Lab 1),0.7312,0.7297,0.7364,0.7330,0.7974
2,Lab 2 model (LogReg),0.7768,0.7824,0.7682,0.7752,0.8305


## Error Analysis (Top 3 Failure Modes)

Below, we inspect where the model fails most often on the validation split.

In [10]:
error_df = split_df["val"][["season", "round", "driver_code", "driver_name", "team_name", "is_top10"]].copy()
error_df["pred"] = val_pred
error_df["proba_top10"] = val_score
error_df["error_type"] = np.where(
    (error_df["is_top10"] == 1) & (error_df["pred"] == 0),
    "False Negative",
    np.where(
        (error_df["is_top10"] == 0) & (error_df["pred"] == 1),
        "False Positive",
        "Correct",
    ),
)

errors_only = error_df[error_df["error_type"] != "Correct"].copy()

print("Failure mode 1: Drivers with most mistakes")
display(errors_only.groupby(["driver_code", "driver_name"]).size().sort_values(ascending=False).head(3).to_frame("errors"))

print("Failure mode 2: Rounds with most mistakes")
display(errors_only.groupby(["season", "round"]).size().sort_values(ascending=False).head(3).to_frame("errors"))

print("Failure mode 3: Team-level concentration of errors")
display(errors_only.groupby("team_name").size().sort_values(ascending=False).head(3).to_frame("errors"))

errors_only.head(10)

Failure mode 1: Drivers with most mistakes


,,errors
driver_code,driver_name,
OCO,Esteban Ocon,12
GAS,Pierre Gasly,11
STR,Lance Stroll,10


Failure mode 2: Rounds with most mistakes


errors
season round        
2023   3           9
       18          9
       21          8

Failure mode 3: Team-level concentration of errors


,errors
team_name,
Alpine,23
Aston Martin,12
McLaren,12


,season,round,driver_code,driver_name,team_name,is_top10,pred,proba_top10,error_type
21,2023,1,ALB,Alexander Albon,Williams,1,0,0.255194,False Negative
28,2023,8,ALB,Alexander Albon,Williams,1,0,0.372321,False Negative
38,2023,18,ALB,Alexander Albon,Williams,1,0,0.252929,False Negative
39,2023,19,ALB,Alexander Albon,Williams,1,0,0.298264,False Negative
41,2023,21,ALB,Alexander Albon,Williams,0,1,0.598064,False Positive
103,2023,15,ALO,Fernando Alonso,Aston Martin,0,1,0.787996,False Positive
109,2023,21,ALO,Fernando Alonso,Aston Martin,1,0,0.455292,False Negative
160,2023,1,BOT,Valtteri Bottas,Alfa Romeo,1,0,0.417672,False Negative
167,2023,8,BOT,Valtteri Bottas,Alfa Romeo,1,0,0.250382,False Negative
170,2023,11,BOT,Valtteri Bottas,Alfa Romeo,0,1,0.529952,False Positive


## Export Draft for comparison_table.md

This cell creates a draft markdown table with your current validation numbers.

In [11]:
def row_to_markdown(label, metrics):
    return (
        f"| {label} | {metrics['Accuracy']:.3f} | {metrics['Precision']:.3f} | "
        f"{metrics['Recall']:.3f} | {metrics['F1']:.3f} | {metrics['ROC-AUC']:.3f} |"
    )

lines = [
    "# Lab 1 vs Lab 2 - Comparison Table",
    "## [Your Name(s)]",
    "| Model / Baseline | Accuracy | Precision | Recall | F1 | ROC-AUC |",
    "|------------------------|----------|-----------|--------|-------|---------|",
    row_to_markdown("Majority class (Lab 1)", majority_val),
    row_to_markdown("Domain heuristic (Lab 1)", heuristic_val),
    "| Prior-period (if done) | 0.000 | 0.000 | 0.000 | 0.000 | 0.000 |",
    row_to_markdown("Lab 2 model (LogReg)", lab2_val_metrics),
    "## Primary metric: [YOUR CHOICE] (justified in Lab 1)",
    "## Interpretation (3-5 sentences)",
    "[Did the model beat the baselines? By how much? Which baseline was hardest to beat? What does this tell you about feature value? If it did not beat a baseline, what would you change next?]",
]

draft_md = "\n".join(lines)
print(draft_md)

out_path = Path("comparison_table.md")
out_path.write_text(draft_md, encoding="utf-8")
print(f"\nDraft written to: {out_path.resolve()}")

# Lab 1 vs Lab 2 - Comparison Table
## [Your Name(s)]
| Model / Baseline | Accuracy | Precision | Recall | F1 | ROC-AUC |
|------------------------|----------|-----------|--------|-------|---------|
| Majority class (Lab 1) | 0.501 | 0.501 | 1.000 | 0.668 | 0.500 |
| Domain heuristic (Lab 1) | 0.731 | 0.730 | 0.736 | 0.733 | 0.797 |
| Prior-period (if done) | 0.000 | 0.000 | 0.000 | 0.000 | 0.000 |
| Lab 2 model (LogReg) | 0.777 | 0.782 | 0.768 | 0.775 | 0.830 |
## Primary metric: [YOUR CHOICE] (justified in Lab 1)
## Interpretation (3-5 sentences)
[Did the model beat the baselines? By how much? Which baseline was hardest to beat? What does this tell you about feature value? If it did not beat a baseline, what would you change next?]

Draft written to: C:\Users\joaqu\OneDrive\Documentos\GitHub\iit414w-lab01-team4\labs\lab_02\comparison_table.md
